# Amazon (AMZN): Donchian Channels

Companion to the [Bubble Analytics article](https://www.bubble-analytics.com/blog/Amazon-AMZN-Backtested-Investment-Strategy-Donchian-Channels), dated 2026-09-30.

Run all cells from this folder after installing `requirements.txt` and JupyterLab.
The included frozen data covers 2016-09-30 to 2026-09-29.
The measured backtest starts on 2017-02-02 after any indicator warmup.
Data files are read locally; no account, API key or network connection is needed to run the backtest.
`TEMPLATE.ipynb` is the existing general strategy template and is separate from this completed example.

The account starts with $100,000. Fees are 0.05% on entry and 0.05% on exit.
Signals are assessed after the close; market entries execute at the next session's open.
Stops and targets, where present, follow the code below. Daily bars cannot establish every intraday price sequence.
Open positions remain marked to market at the end of the sample; closed-trade statistics exclude them.
Spreads, additional slippage, financing and taxes are not modelled.
The benchmark uses the same sample and price convention as the article.

These are historical, in-sample results, not a promise of future performance.

This is the corrected ten-year rerun used in the final article, including entry and exit fees.


In [ ]:
import hashlib
from pathlib import Path
input_hashes = {'AMZN-daily.csv.gz': '0a222ca8c9f13019709411658f164ad92ebee8b281bbdd342c94c778c40d2d4c'}
for filename, expected_hash in input_hashes.items():
    assert hashlib.sha256((Path('data')/filename).read_bytes()).hexdigest() == expected_hash, 'Input file changed: ' + filename


## Strategy and backtest


In [ ]:
from IPython.display import display
"""Amazon Donchian strategy using the last ten years of daily OHLC data."""
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
import backtesting
from backtesting import Backtest, Strategy

ROOT = Path.cwd()
(ROOT/'snapshots').mkdir(exist_ok=True)
(ROOT/'charts').mkdir(exist_ok=True)
raw = pd.read_csv(ROOT/'data/AMZN-daily.csv.gz', parse_dates=['Date']).set_index('Date')
assert raw.index.is_monotonic_increasing and not raw.index.has_duplicates
assert raw.index.max() < pd.Timestamp('2026-09-30')
data = raw[['Open', 'High', 'Low', 'Close']].copy()
# The notebook uses rolling closing-price extremes, not high/low extremes.
data['Up_D'] = data.Close.rolling(86).max()
data['Low_D'] = data.Close.rolling(86).min()
data['Mid_D'] = (data.Up_D + data.Low_D) / 2
data = data.dropna().copy()

def causal_signals(frame):
    upper = (frame.Close.shift(1) < frame.Up_D.shift(2)) & (frame.Close > frame.Up_D.shift(1))
    lower = (frame.Close.shift(1) > frame.Low_D.shift(2)) & (frame.Close < frame.Low_D.shift(1))
    return pd.Series(np.select([upper, lower], [1, 2], default=0), index=frame.index)

data['signal'] = causal_signals(data)
# Check the notebook's loop against the causal version. Its first two rows
# refer to the end of the frame via negative iloc; never carry that into reuse.
legacy = np.zeros(len(data), dtype=int)
for i in range(len(data)):
    if data.Close.iloc[i-1] < data.Up_D.iloc[i-2] and data.Close.iloc[i] > data.Up_D.iloc[i-1]: legacy[i] = 1
    elif data.Close.iloc[i-1] > data.Low_D.iloc[i-2] and data.Close.iloc[i] < data.Low_D.iloc[i-1]: legacy[i] = 2
assert np.array_equal(legacy[2:], data.signal.values[2:])
for end in (200, 1000, len(data)-100):
    assert causal_signals(data.iloc[:end]).equals(data.signal.iloc[:end])

def SIGNAL(): return data.signal
class DonchianAmazon(Strategy):
    def init(self): self.signal = self.I(SIGNAL)
    def next(self):
        price = self.data.Close[-1]
        if self.signal[-1] == 1 and (self.position.is_short or not self.position):
            self.position.close()
            self.buy(size=0.99, tp=1.05*price, tag='Upper breakout')
        elif self.signal[-1] == 2 and (self.position.is_short or not self.position):
            self.position.close()
            self.buy(size=0.99, sl=0.95*price, tp=1.10*price, tag='Lower breakdown')

bt = Backtest(data, DonchianAmazon, cash=100_000, margin=1,
              exclusive_orders=True, commission=0.0005, trade_on_close=False,
              finalize_trades=False)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter('always')
    stats = bt.run()
trades = stats['_trades'].copy()
equity = stats['_equity_curve'].copy()
benchmark = 100_000 * data.Close / data.Close.iloc[0]
benchmark_dd = benchmark / benchmark.cummax() - 1
closed_pnl = float(trades.PnL.sum())
metrics = {str(k): str(v) if isinstance(v,(pd.Timestamp,pd.Timedelta)) else float(v)
           for k,v in stats.items() if not str(k).startswith('_') and isinstance(v,(int,float,np.number,pd.Timestamp,pd.Timedelta))}
metrics.update(commissionPerSide=0.0005,commissionTotalRate=0.001,symbol='AMZN',channelSessions=86,initialCash=100000,
               dataFirst=str(raw.index.min().date()),dataLast=str(raw.index.max().date()),
               benchmarkMaxDrawdownPct=float(benchmark_dd.min()*100),
               closedTradePnL=closed_pnl,closedTradeFinalEquity=100000+closed_pnl,
               openTradesAtEnd=len(stats['_strategy'].trades),
               winningTrades=int((trades.PnL>0).sum()),losingTrades=int((trades.PnL<0).sum()),
               entryGroups={str(tag):{'trades':len(g),'wins':int((g.PnL>0).sum()),'pnl':float(g.PnL.sum())} for tag,g in trades.groupby('Tag')},
               firstTwoSignalCorrections=np.flatnonzero(legacy != data.signal.values).tolist(),
               backtestingVersion=backtesting.__version__,warnings=[str(w.message) for w in captured],
               snapshotSha256=hashlib.sha256((ROOT/'data/AMZN-daily.csv.gz').read_bytes()).hexdigest())
assert len(trades)==int(stats['# Trades'])
assert np.isclose((equity.Equity.iloc[-1]/100000-1)*100, stats['Return [%]'])
assert np.isclose((data.Close.iloc[-1]/data.Close.iloc[0]-1)*100,stats['Buy & Hold Return [%]'])
assert np.isclose((equity.Equity/equity.Equity.cummax()-1).min()*100,stats['Max. Drawdown [%]'])
assert trades.Size.gt(0).all()
assert backtesting.__version__ == '0.6.5', 'Use the pinned strategy runtime for fee-consistent trade statistics'
open_entry_fees = sum(abs(t.size) * t.entry_price * 0.0005 for t in stats['_strategy'].trades)
unrealised_pnl = float(stats['_strategy'].position.pl) - open_entry_fees
assert np.isclose(closed_pnl+100000+unrealised_pnl, equity.Equity.iloc[-1]), 'Realised and unrealised P&L must reconcile to final equity'
metrics['unrealisedPnLAtEnd'] = unrealised_pnl
metrics['openTradeEntryFees'] = open_entry_fees
metrics['openTrades'] = [{'entry':str(t.entry_time),'entryPrice':t.entry_price,'size':t.size,'tag':t.tag} for t in stats['_strategy'].trades]
trades.to_csv(ROOT/'snapshots/trades.csv',index=False)
equity.to_csv(ROOT/'snapshots/equity.csv')
data.to_csv(ROOT/'snapshots/indicators.csv')
benchmark.rename('BenchmarkEquity').to_csv(ROOT/'snapshots/benchmark.csv')
(ROOT/'snapshots/results.json').write_text(json.dumps(metrics,indent=2,allow_nan=False)+'\n')
display(pd.Series({k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}, name='Result'))

# Seeded trade order illustrations use this run's net dollar outcomes.
rng = np.random.default_rng(20260930)
curves = np.array([np.r_[100000, 100000 + np.cumsum(rng.permutation(trades.PnL.to_numpy()))] for _ in range(100)])
assert np.allclose(curves[:, -1], metrics['closedTradeFinalEquity'])
np.save(ROOT/'snapshots/trade-order-curves.npy', curves)
expected_fees = (trades.Size.abs() * (trades.EntryPrice + trades.ExitPrice) * 0.0005)
assert np.allclose(trades.Commission, expected_fees), 'Each trade must charge 0.05% of entry and exit value'


## Charts and account results


In [ ]:
"""Restore Ben's notebook styling, using the verified fee-inclusive results."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
ROOT=Path.cwd()
m=json.loads((ROOT/'snapshots/results.json').read_text())
df=pd.read_csv(ROOT/'snapshots/indicators.csv',parse_dates=['Date']).set_index('Date')
trades=pd.read_csv(ROOT/'snapshots/trades.csv',parse_dates=['EntryTime','ExitTime'])
equity=pd.read_csv(ROOT/'snapshots/equity.csv',parse_dates=['Date']).set_index('Date')
benchmark=pd.read_csv(ROOT/'snapshots/benchmark.csv',parse_dates=['Date']).set_index('Date').BenchmarkEquity
assert np.isclose(100000+trades.PnL.sum(),m['closedTradeFinalEquity'])
GREEN='#00DF9A'
FONT_FAM='Atkinson Hyperlegible, Inter, Helvetica, Arial, sans-serif'
logo=f"<b><span style='color:{GREEN}'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>"
CHARTS={}
def brand(fig,y=1.15):
 fig.update_yaxes(gridcolor='#171717');fig.update_xaxes(gridcolor='#171717')
 fig.add_annotation(text=logo,x=1.0,y=y,xref='paper',yref='paper',xanchor='right',showarrow=False,font=dict(size=12,family=FONT_FAM))
 return fig
df['up_entries']=np.where(df.signal==1,df.Low*(1-.002),np.nan)
df['low_entries']=np.where(df.signal==2,df.Low*(1-.002),np.nan)
bar = max(0, len(df) - 500)
df1 = df[bar:bar+500].copy()

fig = go.Figure(data=[go.Candlestick(x=df1.index,
                open=df1['Open'],
                high=df1['High'],
                low=df1['Low'],
                close=df1['Close'],
                increasing_line_color = 'rgba(19,156,19,0.8)',
                decreasing_line_color = 'rgba(175,07,49,0.8)',
                name = 'AMZN')])

fig.add_scatter(x=df1.index, y=df1['up_entries'], mode="markers",
                marker=dict(size=7, color="white"),
                name="Trade Signals")

fig.add_scatter(x=df1.index, y=df1['low_entries'], mode="markers",
                marker=dict(size=7, color="gold"),
                name="Trade Signals")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.Up_D, 
                         opacity=0.8, 
                         line=dict(color='white', width=1),
                        name='Upper Donchian'))

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.Low_D, 
                         opacity=0.6, 
                         line=dict(color='white', width=1),
                        name='Lower Donchian'))

fig.update_layout(
    annotations=[dict(text="AMZN with Donchian Channels",
                    font=dict(color="white", size=14),
                    xref="paper",
                    yref="paper",
                    x=0.10,
                    y=1.05,
                    showarrow=False)])

fig.update_layout(autosize=False, width=1000, height=660, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

GREEN     = "#00DF9A"   
FONT_FAM  = "Atkinson Hyperlegible, Inter, Helvetica, Arial, sans-serif"
BRAND_CORNER = "right"  

bubble_logo = (
    f"<b><span style='color:{GREEN}'>BUBBLE</span>"
    f"<span style='color:white'>ANALYTICS</span></b>"
)

fig.add_annotation(
    text=bubble_logo,
    x=1.0 if BRAND_CORNER == "right" else 0.0,
    y=1.05, xref="paper", yref="paper",
    xanchor="right" if BRAND_CORNER == "right" else "left",
    showarrow=False,
    font=dict(size=12, family=FONT_FAM)
)

#fig.write_image('chart.png', scale=2)



CHARTS['chart-donchian']=fig

curves=np.load(ROOT/'snapshots/trade-order-curves.npy')
assert curves.shape==(100,len(trades)+1) and np.allclose(curves[:,-1],m['closedTradeFinalEquity'])
mc=go.Figure()
for curve in curves:mc.add_trace(go.Scatter(y=curve-100000,mode='lines',opacity=.6,showlegend=False))
losses_first = np.r_[0, np.cumsum(np.r_[trades.PnL[trades.PnL<0],trades.PnL[trades.PnL>=0]])]
mc.add_trace(go.Scatter(y=losses_first,mode='lines',name='All losing trades first',line=dict(color='#ff5265',width=3,dash='dash')))
mc.add_hline(y=0,line_color='#a6b0bb',line_width=1)
mc.update_layout(title='Donchian Strategy Monte Carlo Simulation',xaxis_title='Trade Number',yaxis_title='Cumulative profit ($)',template='plotly_dark',autosize=False,width=1000,height=600)
CHARTS['chart-monte-carlo']=brand(mc)

pnl=go.Figure(go.Scatter(x=trades.ExitTime,y=trades.PnL.cumsum(),mode='lines',name='Cumulative PnL',line=dict(color='#00df9a')))
pnl.update_layout(title='Donchian Strategy PnL',template='plotly_dark',autosize=False,width=1000,height=600)
CHARTS['chart-pnl']=brand(pnl)

metrics=['Start','End','# Trades','Exposure Time [%]','Win Rate [%]','Return [%]','Buy & Hold Return [%]','Max. Drawdown [%]','Avg. Drawdown [%]','Sharpe Ratio','Sortino Ratio','Calmar Ratio','Best Trade [%]','Worst Trade [%]','Avg. Trade [%]','Profit Factor','Expectancy [%]']
values=[]
for key in metrics:
 value=m[key]
 if key in ('Start','End'):value=value.split(' ')[0]
 elif key=='# Trades':value=str(int(value))
 else:value=f'{value:.2f}'
 values.append(value)
SITE_BG='#12181B';CARD_BG='#161D21';BORDER='rgba(255,255,255,0.08)';TEXT='#E6ECEF'
stripe=['#141A1E' if i%2==0 else '#101519' for i in range(len(metrics))]
stats=go.Figure(go.Table(columnwidth=[1,1],header=dict(values=['',''],height=1,fill_color=CARD_BG,line_color=BORDER,font=dict(size=1)),cells=dict(values=[[f'<b>{x}</b>' for x in metrics],[f'<b>{x}</b>' for x in values]],align=['left','right'],height=30,font=dict(size=13,color=TEXT,family=FONT_FAM),fill_color=[stripe,stripe],line_color=BORDER,line_width=1)))
stats.update_layout(title=dict(text='Donchian Strategy Stats',font=dict(size=16,color=TEXT,family=FONT_FAM),x=.5,xanchor='right'),width=500,height=700,paper_bgcolor=SITE_BG,plot_bgcolor=CARD_BG,margin=dict(l=30,r=30,t=90,b=30))
stats.add_annotation(text=logo,x=1,y=1.11,xref='paper',yref='paper',xanchor='right',showarrow=False,font=dict(size=12,family=FONT_FAM))
stats.add_shape(type='rect',xref='paper',yref='paper',x0=0,y0=0,x1=1,y1=1,line=dict(color=BORDER,width=1),fillcolor=CARD_BG,layer='below')
CHARTS['stats-comparison']=stats
# Carry the additional return/drawdown comparisons into the same visual style.
for name,title,ylabel,series in [
 ('chart-equity','Donchian Strategy vs AMZN Buy and Hold','Return (%)',[(equity.Equity/100000-1)*100,(benchmark/100000-1)*100]),
 ('chart-drawdown','Donchian Strategy Drawdown','Drawdown (%)',[(equity.Equity/equity.Equity.cummax()-1)*100,(benchmark/benchmark.cummax()-1)*100])]:
 f=go.Figure()
 for values,label,color in zip(series,['Strategy after fees','AMZN buy and hold'],['#00df9a','#a6b0bb']):
  f.add_scatter(x=values.index,y=values,name=label,mode='lines',line=dict(color=color))
 f.update_layout(title=title,yaxis_title=ylabel,template='plotly_dark',autosize=False,width=1000,height=600)
 CHARTS[name]=brand(f)

# Consistent branding with a dedicated header above titles and legends.
wordmark = "<b><span style='color:#00DF9A'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>"
for figure in CHARTS.values():
    figure.layout.annotations = tuple(a for a in figure.layout.annotations if 'BUBBLE' not in (a.text or ''))
    width, height = figure.layout.width or 1100, figure.layout.height or 700
    figure.update_layout(width=width, height=height, margin_t=max(figure.layout.margin.t or 0, 150))
    if figure.layout.title.text:
        figure.update_layout(title_y=1-65/height, title_yanchor='top')
    inner_height = height - figure.layout.margin.t - (figure.layout.margin.b or 80)
    figure.add_annotation(text=wordmark, x=1, y=1+(figure.layout.margin.t-20)/inner_height,
        xref='paper', yref='paper', xanchor='right', yanchor='top', showarrow=False,
        font=dict(size=12, family='Arial, Helvetica, sans-serif'))


In [ ]:
for figure in CHARTS.values():
    display(figure)
